# BIST harness: Gemma 3 4B QLoRA eğitimi (Kaggle)
**Ayarlar (sağ panel, Settings):** Accelerator = **GPU T4 x2** (P100 SEÇMEYİN: Unsloth desteklemez) · Internet = **On** · Persistence = Files only.
**Girdi:** sağ panelde *Input → + Add Input* ile `colab_bundle.zip` içeren dataset'i ekleyin.
**Akış:** hücre 1-4'ü elle çalıştırın (hata varsa erken görürsünüz). 6. hücreyi elle ÇALIŞTIRMAYIN: sağ üstten **Save Version → Save & Run All (Commit)** deyin; eğitim tarayıcı kapalıyken arka planda sürer (~4 saat). Sonuç: sürüm sayfası → **Output** → `result/`.
Ayrıntı: `training/README.md`. Bu not defteri Kaggle'da denenmemiştir; hata verirse hata metnini paylaşın.

In [ ]:
!nvidia-smi
import torch, shutil
print("CUDA:", torch.cuda.is_available(), "| GPU sayısı:", torch.cuda.device_count(), "| bf16:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else None)
for p in ("/kaggle/working", "/tmp"):
    print(p, "boş GB:", round(shutil.disk_usage(p).free / 1e9, 1))   # /kaggle/working sınırı 20 GB: büyük GGUF ara dosyaları /tmp'de tutulur

In [ ]:
!pip install -q unsloth

In [ ]:
import glob, os, shutil, zipfile
WORK = "/kaggle/working/bundle"
os.makedirs(WORK, exist_ok=True)
zips = glob.glob("/kaggle/input/**/colab_bundle.zip", recursive=True)
if zips:                                  # dataset zip olarak kaldıysa aç
    zipfile.ZipFile(zips[0]).extractall(WORK)
else:                                     # Kaggle zip'i otomatik açtıysa klasörü kopyala
    trains = glob.glob("/kaggle/input/**/train.jsonl", recursive=True)
    assert trains, "Dataset ekli değil: sağ panelde + Add Input ile colab_bundle dataset'ini ekleyin"
    shutil.copytree(os.path.dirname(trains[0]), WORK, dirs_exist_ok=True)
os.chdir(WORK)
print(sorted(os.listdir(".")), "| betik var mı:", os.path.exists("training/colab/train_qlora.py"))

## Kuru çalıştırma (model yüklemeden): kaç örnek 8192 token sınırına sığıyor, eğitilen token payı

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python training/colab/train_qlora.py --train train.jsonl --val val.jsonl --out /tmp/outputs/v2 --dry-run

## Eğitim (YALNIZCA Save & Run All ile)
`--out /tmp/...`: büyük ara dosyalar (birleştirilmiş 16-bit model, f16 GGUF) /kaggle/working'in 20 GB sınırına girmesin. Kontrol noktaları /kaggle/working/ckpt'ye yazılır. Adım başına ~153 sn x 75 adım ≈ 3,2 saat + GGUF dışa aktarma.

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python training/colab/train_qlora.py --train train.jsonl --val val.jsonl --out /tmp/outputs/v2 --epochs 1 --gguf q4_k_m --save-steps 5 --checkpoint-dir /kaggle/working/ckpt --resume

## Sonuçları topla (eğitim ya da GGUF kısmen başarısız olsa bile çalışır)

In [ ]:
import glob, os, shutil
O, R = "/tmp/outputs/v2", "/kaggle/working/result"
os.makedirs(R, exist_ok=True)
for name in ("data_stats.json", "train_result.json"):
    if os.path.exists(f"{O}/{name}"):
        shutil.copy(f"{O}/{name}", R)
if os.path.isdir(f"{O}/adapter"):
    shutil.copytree(f"{O}/adapter", f"{R}/adapter", dirs_exist_ok=True)
ggufs = [g for root in (O, "/kaggle/working/bundle") for g in glob.glob(f"{root}/**/*.gguf", recursive=True) if "q4_k_m" in os.path.basename(g).lower()]
for g in ggufs:
    shutil.copy(g, R)
print("GGUF:", ggufs or "BULUNAMADI (adaptör kaydedildi; GGUF'u ayrıca üretmek gerekir)")
shutil.rmtree("/kaggle/working/ckpt", ignore_errors=True)
shutil.rmtree("/kaggle/working/bundle", ignore_errors=True)
!ls -la /kaggle/working/result
!cat /kaggle/working/result/train_result.json